In [2]:
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn

print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"joblib: {joblib.__version__}")

Python: 3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]
NumPy: 2.3.4
pandas: 2.3.3
scikit-learn: 1.7.2
joblib: 1.5.2


In [3]:
SEED = 2729

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"Train columns: {train.columns.tolist()}")
print(f"Test columns:  {test.columns.tolist()}")
print(f"Columns present in train but missing from test: {train.columns.difference(test.columns).tolist()}")
print(f"Share of 'yes' in train: {(train['y'] == 'yes').mean():.4f}")
print(f"Duplicate train id values: {train['id'].duplicated().sum()}")
print(f"Duplicate test id values:  {test['id'].duplicated().sum()}")

Train shape: (32940, 22)
Test shape:  (8236, 20)
Train columns: ['id', 'age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']
Test columns:  ['id', 'age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Columns present in train but missing from test: ['duration', 'y']
Share of 'yes' in train: 0.1127
Duplicate train id values: 0
Duplicate test id values:  0


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

y_train = train["y"].eq("yes").astype(int)
dummy_X = np.zeros((len(y_train), 1))

always_no_predictions = np.zeros(len(y_train), dtype=int)
prior_model = DummyClassifier(strategy="prior", random_state=SEED)
prior_model.fit(dummy_X, y_train)
prior_predictions = prior_model.predict(dummy_X)

baseline_results = pd.DataFrame(
    [
        {
            "baseline": "Always no",
            "accuracy": accuracy_score(y_train, always_no_predictions),
            "f1": f1_score(y_train, always_no_predictions, zero_division=0),
        },
        {
            "baseline": "DummyClassifier (prior)",
            "accuracy": accuracy_score(y_train, prior_predictions),
            "f1": f1_score(y_train, prior_predictions, zero_division=0),
        },
    ]
).set_index("baseline")
display(baseline_results.round(3))

always_no = baseline_results.loc["Always no"]
if always_no["accuracy"] > 0.8 and always_no["f1"] == 0:
    print("Conclusion: always predicting 'no' gives high accuracy but misses every 'yes' (F1 = 0).")
else:
    print("Conclusion: compare F1 as well as accuracy; accuracy alone does not show performance on 'yes'.")

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

from feature_transformers import add_was_contacted

feature_cols = [column for column in train.columns if column not in {"id", "duration", "y"}]
train_features = train.loc[:, feature_cols]

numeric_type_cols = train_features.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_features.select_dtypes(exclude=np.number).columns.tolist()
type_summary = pd.DataFrame(
    {
        "feature_count": [len(numeric_type_cols), len(categorical_cols)],
        "columns": [", ".join(numeric_type_cols), ", ".join(categorical_cols)],
    },
    index=["Numeric", "Categorical"],
)
display(type_summary)

unknown_counts = train_features.eq("unknown").sum()
display(unknown_counts[unknown_counts.gt(0)].rename("unknown_count").to_frame())
print(f"Share of pdays == 999: {train_features['pdays'].eq(999).mean():.4f}")

# Exclude the pdays sentinel from numeric input; derive was_contacted inside the pipeline.
numeric_cols = [column for column in numeric_type_cols if column != "pdays"] + ["was_contacted"]
print(f"numeric_cols: {numeric_cols}")
print(f"categorical_cols: {categorical_cols}")

feature_engineering = Pipeline(
    [
        (
            "add_was_contacted",
            FunctionTransformer(add_was_contacted, validate=False),
        )
    ]
)

,accuracy,f1
baseline,,
Always no,0.887,0.0
DummyClassifier (prior),0.887,0.0


Conclusion: always predicting 'no' gives high accuracy but misses every 'yes' (F1 = 0).
